In [1]:
from IPython.display import clear_output
!pip install pip3-autoremove
!pip-autoremove torch torchvision torchaudio -y
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu121
!pip install unsloth
# Also get the latest nightly Unsloth!
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git
!pip install --upgrade --no-cache-dir transformers
clear_output()

In [ ]:
import huggingface_hub

huggingface_hub.login(token="hf_XXXXXXXXXXXXXXXXXXXXX")

In [3]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from datasets import Dataset
import json


In [4]:
def load_model(model_name="meta-llama/Llama-3.2-3B-Instruct"):
    # bnb_config = BitsAndBytesConfig(
    #     load_in_4bit=True,
    #     bnb_4bit_quant_type="nf4",
    #     bnb_4bit_compute_dtype=torch.float16,
    #     bnb_4bit_use_double_quant=True,
    # )
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token  # Set padding token
    
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        # quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16
    )
    
    return model, tokenizer

In [5]:
def generate_response(model, tokenizer, prompt, max_length=200):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_length,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.pad_token_id
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [6]:
def read_dataset(ds_path):
    dataset = []
    with open(ds_path, 'r') as f:
        for line in f:
            try:
                conversation = json.loads(line)
                formatted_text = tokenizer.apply_chat_template(
                    conversation,
                    tokenize=False,
                    add_generation_prompt=False
                )
                dataset.append({"text": formatted_text})
            except:
                continue
    return dataset

In [7]:
def train_model(model, tokenizer, train_dataset, eval_dataset=None):
    # Tokenization
    def tokenize_function(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            max_length=512,  # Use longer context
            padding=True  # Dynamic padding via collator
        )

    tokenized_train = train_dataset.map(tokenize_function, batched=True)
    # if eval_dataset:
        # tokenized_eval = eval_dataset.map(tokenize_function, batched=True)

    # Data Collator
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False  # Causal language modeling
    )

    # Training Arguments
    training_args = TrainingArguments(
        output_dir="./full_finetune_results",
        per_device_train_batch_size=1,  # Reduce if OOM
        # per_device_eval_batch_size=1,
        num_train_epochs=1,
        learning_rate=1e-5,  # Lower LR for full fine-tuning
        weight_decay=0.01,
        warmup_ratio=0.1,
        gradient_accumulation_steps=1,  # Critical for memory management
        gradient_checkpointing=True,  # Reduce memory usage
        fp16=False,  # Use mixed precision
        evaluation_strategy="no",
        # eval_steps=20,
        save_strategy="steps",
        save_steps=20,
        logging_steps=20,
        report_to="none",
        optim="adafactor",  # Better for large models
        remove_unused_columns=True,
        bf16=False
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=None,
        data_collator=data_collator,
    )

    # Start training
    trainer.train()
    
    # Save final model
    model.save_pretrained("./fully_finetuned_llama")
    tokenizer.save_pretrained("./fully_finetuned_llama")
    
    return trainer


In [8]:
def plot_loss(trainer):
    history = trainer.state.log_history
    train_loss = [x['loss'] for x in history if 'loss' in x]
    eval_loss = [x['eval_loss'] for x in history if 'eval_loss' in x]
    
    plt.figure(figsize=(10, 6))
    plt.plot(train_loss, label='Training Loss')
    if eval_loss:
        plt.plot(eval_loss, label='Validation Loss')
    plt.xlabel('Steps')
    plt.ylabel('Loss')
    plt.title('Training Progress')
    plt.legend()
    plt.show()

In [9]:
model, tokenizer = load_model()

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [10]:
!wget https://huggingface.co/datasets/miladmim/slim-orca-dedup-chat-50k-persian/resolve/main/data.jsonl?download=true

--2025-02-02 20:01:38--  https://huggingface.co/datasets/miladmim/slim-orca-dedup-chat-50k-persian/resolve/main/data.jsonl?download=true
Resolving huggingface.co (huggingface.co)... 3.171.171.104, 3.171.171.128, 3.171.171.65, ...
Connecting to huggingface.co (huggingface.co)|3.171.171.104|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://cdn-lfs-us-1.hf.co/repos/03/91/039119bf3b6f3c083e8207d297da2d654c372ba1b6778664345655e6e4af7aa5/3ca27a9b4e6f28661e2ac780ba648b1cf983c2e4508ceba00ab86dc3b2d465f3?response-content-disposition=attachment%3B+filename*%3DUTF-8%27%27data.jsonl%3B+filename%3D%22data.jsonl%22%3B&Expires=1738530098&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTczODUzMDA5OH19LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy11cy0xLmhmLmNvL3JlcG9zLzAzLzkxLzAzOTExOWJmM2I2ZjNjMDgzZTgyMDdkMjk3ZGEyZDY1NGMzNzJiYTFiNjc3ODY2NDM0NTY1NWU2ZTRhZjdhYTUvM2NhMjdhOWI0ZTZmMjg2NjFlMmFjNzgwYmE2NDhiMWNmOTgzYzJlNDUwOGNlYmEwMGFiODZkY

In [15]:
dataset_path = "/kaggle/working/data.jsonl?download=true"
full_dataset = Dataset.from_list(read_dataset(dataset_path))
subset_dataset = full_dataset.select(range(100))
split_dataset = subset_dataset.train_test_split(test_size=0.1, seed=42)

In [12]:
for param in model.model.layers[0].parameters():
    # param.data = param.data.float()
    param.requires_grad = True

# Re-enable (unfreeze) the parameters of the last transformer layer.
for param in model.model.layers[-1].parameters():
    # param.data = param.data.float()
    param.requires_grad = True

    # Optionally, if you want the language modeling head to be trainable too:
# for param in model.lm_head.parameters():
    # param.data = param.data.float()
    # param.requires_grad = True

In [13]:
torch.cuda.empty_cache()

In [16]:
trainer = train_model(
    model,
    tokenizer,
    split_dataset["train"],
    split_dataset["test"]
)
plot_loss(trainer)

Map:   0%|          | 0/90 [00:00<?, ? examples/s]

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Step,Training Loss
20,2.411300
40,1.917300
60,1.901500


SafetensorError: Error while serializing: IoError(Os { code: 28, kind: StorageFull, message: "No space left on device" })

In [17]:
prompt = "یک داسنان کوتاه در مورد دانشجوی دانشگاه تهران تعریف کن"
print("Base model response:")
print(generate_response(model, tokenizer, prompt))

Base model response:


/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


یک داسنان کوتاه در مورد دانشجوی دانشگاه تهران تعریف کنید.
به عنوان یک داسنان، من به دلیل اینکه من یک دانشجوی دانشگاه تهران بودم، برای اولین بار در سال 1380 با یک داسنان نامزدی acquaintance Meeting را از دست دادم. این یک experience was که من با یک داسنان به عنوان یک دانش آموز در دانشگاه تهران، به عنوان یک student، به عنوان یک student، به عنوان یک student، به عنوان یک student، به عنوان یک student، به عنوان یک student، به عنوان یک student، به عنوان یک student، به عنوان یک student، به عنوان یک student، به عنوان یک student، به عنوان یک student، به عنوان یک student، به عنوان یک student، به عنوان یک student، به عنوان یک student، به عنوان یک student، به عنوان یک student، به عنوان یک student، به عنوان یک student، به عنوان یک student، به عنوان یک student، به عنوان یک student، به عنوان یک student، به عنوان یک student، به عنوان یک student، به عنوان یک student،


In [19]:
model.eval()

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((3072,), eps=1e-05)
    (rotary_emb